<div style="text-align: justify; max-width: 90ch">

# Lecture 01a: Talking to a local model: tokens, context, temperature

**Status: Mandatory reading.**

In the Python course you trained models yourself: `fit`, `predict`, a metric. A **large language model (LLM)** is different:
someone else trained it for months on thousands of GPUs, and your job is to *run* it and *talk* to it. This notebook makes that concrete on your own laptop, with no account and no API key.

**What this notebook covers**

- **Ollama**, a local model server, and its Python client.
- What running locally gains and costs.
- Your first call from Python: what you send, what the reply contains.
- **Tokens**: the unit a model reads, writes, and is paid in.
- The **context window**: how much text a model can see at once.
- **Temperature**: why the same question gives different answers.

</div>

<div style="text-align: justify; max-width: 90ch">

Before you start, finish the quick starts of the guides `01a_git_uv` to `01c_ollama` in this lecture's `infra_tools/` folder,
which we run together in class:

- `uv sync`
- Ollama installed
- `ollama pull qwen3:1.7b`

The rest of each guide explains the tools themselves: `uv` and the course environment (`01a_git_uv`), and how the Ollama server, its address and its API work (`01c_ollama`).
This notebook is about the model.

</div>

In [1]:
import os
from pathlib import Path

import ollama
import pandas as pd
from dotenv import load_dotenv
from transformers import AutoTokenizer

<div style="text-align: justify; max-width: 90ch">

## 0. The configuration cell

Every notebook in this course starts with the same cell. You edit **one line**: `TIER`, which picks the chat model.

### The model that we use and where it runs

- **`cpu`** uses [`qwen3:1.7b`](https://ollama.com/library/qwen3), a model small enough for any laptop.
- **`gpu`** uses `qwen3:8b`, a bigger sibling, for a GPU with at least 8 GB of **VRAM**: the GPU's own memory, separate from the computer's RAM.

Which fits *your* machine? `llmfit plan "Qwen/Qwen3-8B" --quant Q4_K_M --context 8192` tells you (guide `01b_llmfit`), and `lec_01b` explains what else goes into that choice.

</div>

<div style="text-align: justify; max-width: 90ch">

### The personal settings in the `.env` file

The cell also reads **`.env`**, your personal settings file, with `load_dotenv`, which copies each line into an **environment variable**:
a named value the operating system keeps for programs, outside your code. `os.environ.get` reads each setting back, with a default for when `.env` does not set it.
Guide `01d_env_hugging_face` explains the file and the commented-out override lines. `OLLAMA_HOST` is the address of Ollama (section 1).
`MLFLOW_URI` and `EXPERIMENT` wait for lecture 2, where MLflow enters; they are here because the cell is the same in every notebook.

The last line calls a helper from `llm_course`, the small package installed by `uv sync`. It checks that Ollama answers and that the models are pulled, and if not it prints the exact command to run.

</div>

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. Ollama, a local model server

A model is a large file of numbers, up to several gigabytes: its **weights**, the numbers learned in training. **Open models**, such as the Qwen3 family this course uses, publish their weights for anyone to download;
`ollama pull` did that for you.

For a model to answer, something must load the weights into memory and compute with them. [**Ollama**](https://docs.ollama.com/) does this as a **server**:
a program that runs in the background, keeps the model in memory, and answers requests at the address in `OLLAMA_HOST`. Your code is the **client**:
it sends a request to the server and gets a response back. Guide `01c_ollama` explains the address, its port, and the **API**, the set of requests the server understands.

</div>

<div style="text-align: justify; max-width: 90ch">

The `ollama` Python package is the code for the client. It runs no model: it turns a Python call into a request to the server,
gets the response back, and turns it into a Python object whose fields you read with a dot, like `df.shape`.

First we create the Python client object with `ollama.Client(host=OLLAMA_HOST)`, which stores the address, and then `client.list()` asks which models are on disk.

</div>

In [3]:
client = ollama.Client(host=OLLAMA_HOST)

models = client.list().models

pd.DataFrame(
    {
        "model": [m.model for m in models],
        "size_gb": [round(m.size / 1e9, 2) for m in models],
        "parameters": [m.details.parameter_size for m in models],
        "quantization": [m.details.quantization_level for m in models],
    }
)

,model,size_gb,parameters,quantization
0,nomic-embed-text:latest,0.27,137M,F16
1,gemma3:4b,3.34,4.3B,Q4_K_M
2,qwen3:8b,5.23,8.2B,Q4_K_M
3,qwen3:0.6b,0.52,751.63M,Q4_K_M
4,qwen3:1.7b,1.36,2.0B,Q4_K_M


<div style="text-align: justify; max-width: 90ch">

Your table lists the models you pulled. Three columns deserve a first look:

- **parameters**: the number of weights. The coefficients of a scikit-learn `LinearRegression` are weights too, a handful of them.
  `qwen3:1.7b` shows `2.0B`: the model uses one table of 311 million weights twice, at its input and at its output, and its name counts it once while the file stores two copies.
- **quantization**: how many bits each weight is stored in. `Q4_K_M` stores most weights in 4 bits instead of the original 16,
  which is why 2 billion weights take 1.4 GB, not 4 GB. `lec_01b` returns to this.
- **size_gb**: what that costs on disk, and roughly in memory when the model is loaded.

[`nomic-embed-text`](https://ollama.com/library/nomic-embed-text) is not a chat model; lecture 2 uses it to turn text into numbers.

</div>

<div style="text-align: justify; max-width: 90ch">

### Local or hosted?

The alternative is a **hosted model**: a company runs it on its servers, and your client sends each request over the internet with an **API key**,
a secret that identifies your account. Running locally gives:

- **Privacy**: nothing you send leaves the machine, which matters for a company's documents.
- **No account, no bill per request**, and it works offline.
- **Stability**: a pulled model stays the same until you pull it again; a provider can change or retire a hosted one.

The price is **quality and speed**: the model must fit your memory, so it is smaller and makes more mistakes than the largest hosted ones,
and your hardware sets its speed.

Hosted wins when:

- the data may leave
- the task needs the strongest model
- no machine fits

`lec_01b` turns this into requirements; the optional `lec_01f` calls a hosted model.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Your first call

An LLM does one thing: given text, it predicts the next **token** (a word piece, section 3), appends it, and repeats until it emits a **stop token**,
a special token that means "done". Everything else, chat, coding, summarising, is that loop with a clever **prompt**, the text you send.

`client.chat` sends one request to `/api/chat`. Its arguments:

- **`model`**: which pulled model answers.
- **`messages`**: the conversation, a list of dicts, each with a `role` and `content`. For now there is one message, from you, the `user`;
  `lec_01c` adds the other roles.
- **`think=False`**: `qwen3:1.7b` is a **reasoning model**: by default it "thinks out loud" before answering, which is slow.
  This switches that off; `lec_01b` shows what thinking costs.
- **`options`**: generation settings ([full list](https://docs.ollama.com/modelfile#valid-parameters-and-values)), here `temperature` (section 5) and `num_predict`,
  the most tokens the answer may have.

</div>

In [4]:
response = client.chat(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is data leakage in machine learning?",
        }
    ],
    think=False,
    options={
        "temperature": 0,
        "num_predict": 150,
    },  # num_predict caps the answer length
)

print(response.message.content)

Data leakage in machine learning refers to a situation where a model inadvertently learns relationships between the training data and the target variable that are not generalizable to new, independent data, often due to leakage of information from the test set into the training set.


<div style="text-align: justify; max-width: 90ch">

Read the answer carefully and check it against the definition: data leakage means that information from outside the training data, such as the test set, is used to build the model.
A model's answer can sound fluent and confident and still be inaccurate or incomplete: a fluent answer is not necessarily a correct one;
lecture 3 measures how often answers are right.

</div>

<div style="text-align: justify; max-width: 90ch">

The `response` carries more than the text. Read the fields of Ollama's [chat response](https://docs.ollama.com/api/chat):

| Field | What it holds |
| --- | --- |
| `message.content`, `message.role` | the answer, and who wrote it: `assistant` |
| `done_reason` | why generation stopped: `stop` (the model finished) or `length` (it hit `num_predict`) |
| `prompt_eval_count` | tokens the model read |
| `eval_count` | tokens it wrote |
| `eval_duration`, `total_duration` | time in nanoseconds (billionths of a second, hence `/ 1e9`); the total includes loading the model |

"eval" here means the model's computation, not an evaluation of quality.

</div>

In [5]:
tokens_in = response.prompt_eval_count
tokens_out = response.eval_count
seconds_out = response.eval_duration / 1e9

print(f"prompt tokens : {tokens_in}")
print(f"answer tokens : {tokens_out}")
print(
    f"generation    : {seconds_out:.1f} s  ->  {tokens_out / seconds_out:.1f} tokens/s"
)
print(
    f"total         : {response.total_duration / 1e9:.1f} s (includes loading the model the first time)"
)

prompt tokens : 28
answer tokens : 50
generation    : 0.3 s  ->  158.8 tokens/s
total         : 2.6 s (includes loading the model the first time)


<div style="text-align: justify; max-width: 90ch">

The prompt counts more tokens than the 12-token question (section 3 shows how to count): 28 in one example run, so Ollama added 16.
They are markers for where a message starts and ends and who wrote it, plus a switch that turns thinking off. They take time and space like any other token.

Tokens per second is the number to remember when you compare models or machines. In example runs, `qwen3:1.7b` wrote 10-25 tokens/s on a laptop CPU and over 100 tokens/s on a GPU.

### ⏸ Pause point

Run `client.ps()` below and compare with your neighbours: is the model on the CPU or the GPU, and how much memory does it use?

</div>

In [6]:
loaded = client.ps().models

pd.DataFrame(
    {
        "model": [m.model for m in loaded],
        "memory_gb": [round(m.size / 1e9, 2) for m in loaded],
        "on_gpu_gb": [round(m.size_vram / 1e9, 2) for m in loaded],
    }
)

,model,memory_gb,on_gpu_gb
0,qwen3:1.7b,1.7,1.7


<div style="text-align: justify; max-width: 90ch">

`on_gpu_gb` equal to `memory_gb` means the whole model sits on a GPU; `0.0` means it runs on the CPU; a value in between means it did not fit and is split, which is slower.

A loaded model takes more memory than its file. The next cell compares the two.

</div>

In [7]:
file_gb = {m.model: m.size for m in models}[CHAT_MODEL] / 1e9
memory_gb = {m.model: m.size for m in loaded}[CHAT_MODEL] / 1e9

print(f"{CHAT_MODEL}: file {file_gb:.2f} GB, in memory {memory_gb:.2f} GB")
print(f"extra in memory: {memory_gb - file_gb:.2f} GB")

qwen3:1.7b: file 1.36 GB, in memory 1.70 GB
extra in memory: 0.34 GB


<div style="text-align: justify; max-width: 90ch">

The extra memory holds the context window (section 4) and working space for the computation. If the cell raises `KeyError`, the model was unloaded in the meantime:
Ollama keeps a model loaded for five minutes after the last call ([FAQ](https://docs.ollama.com/faq)), which is also why the second call is faster than the first.
Run the chat cell of section 2 and these two cells again.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Tokens

A model reads numbers, not letters. A **tokenizer** cuts text into **tokens**, each a word, a word piece, or a punctuation mark,
and replaces each with its **token ID**, its position in the **vocabulary**: the fixed list of every token the model knows, about 151 000 for Qwen3.
The vocabulary is fixed before training, so a tokenizer belongs to its model family. Tokens matter three times over:

- **Cost**: hosted models charge per token; local models spend time per token.
- **Capacity**: the context window (next section) is measured in tokens.
- **Speed**: generation is one token at a time.

</div>

<div style="text-align: justify; max-width: 90ch">

To count tokens without running the model, we load the tokenizer alone with [`transformers`](https://huggingface.co/docs/transformers/index),
Hugging Face's Python library for models. `AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")` downloads the tokenizer files (about 16 MB,
one-time) from the model's page on the **Hugging Face Hub**, the website where model makers publish their files ([`Qwen/Qwen3-1.7B`](https://huggingface.co/Qwen/Qwen3-1.7B);
`lec_01b` explores the Hub). It is the tokenizer that Ollama's `qwen3:1.7b` uses. `encode` turns text into token IDs; `convert_ids_to_tokens` shows the piece behind each ID.

</div>

In [8]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

english = "Data leakage happens when the test set influences training."

ids = tokenizer.encode(english)

print(len(english), "characters ->", len(ids), "tokens")

pd.DataFrame({"token": tokenizer.convert_ids_to_tokens(ids), "token_id": ids}).T

59 characters -> 10 tokens


,0,1,2,3,4,5,6,7,8,9
token,Data,Ġleakage,Ġhappens,Ġwhen,Ġthe,Ġtest,Ġset,Ġinfluences,Ġtraining,.
token_id,1043,80273,8573,979,279,1273,738,33353,4862,13


<div style="text-align: justify; max-width: 90ch">

`Ġ` marks a token that starts with a space: `Ġleakage` is " leakage". Every word of this sentence is a single token: 59 characters, 10 tokens, 5.9 characters per token.
A common rule of thumb for English text is about 4 characters per token; a sentence of common, whole words does better. Watch what happens to the same sentence in Greek.

</div>

In [9]:
greek = "Διαρροή δεδομένων συμβαίνει όταν το test set επηρεάζει την εκπαίδευση."

rows = []

for label, text in [("English", english), ("Greek", greek)]:
    n_tokens = len(tokenizer.encode(text))
    rows.append(
        {
            "language": label,
            "characters": len(text),
            "tokens": n_tokens,
            "chars_per_token": round(len(text) / n_tokens, 1),
        }
    )

pd.DataFrame(rows)

,language,characters,tokens,chars_per_token
0,English,59,10,5.9
1,Greek,70,57,1.2


<div style="text-align: justify; max-width: 90ch">

Greek takes almost six times as many tokens (57 against 10) for the same sentence. A vocabulary is built from the frequent pieces of the training text,
where Greek is rare, so the tokenizer spells most Greek words letter by letter: `Διαρροή` alone is seven tokens. The same piece always gets the same ID:
both `ρ` of `Διαρροή` are token 38079. Two consequences you will meet again:

- **Capacity and speed**: a Greek prompt fills the context window almost six times faster, and a Greek answer takes almost six times longer to generate.
- **Model choice**: which model to pick for a Greek use case is partly a **tokenizer** question (`lec_01b`).

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. The context window

You can reread page 1 of a long report while reading page 40. A model cannot: it computes over a fixed number of tokens at once, and text outside that span does not exist for it.
The **context window** is that maximum number of tokens: your messages plus its answer.

You set the window per call with Ollama's option `num_ctx`. A bigger window costs memory, because Ollama reserves space for every token of it when it loads the model:
in example runs, `qwen3:1.7b` took 1.7-1.9 GB with a 4096-token window and about 0.5 GB more with 8192.

Let's put a fact at the *start* of a long text and ask about it. First we make sure the text is longer than the window we will use.

</div>

In [10]:
needle = "The secret word for this lecture is PELICAN."

filler = " ".join(
    f"Note {i}: in the Python course we used pandas for tables, seaborn for plots, and scikit-learn for models."
    for i in range(60)
)

long_text = needle + " " + filler

question = "What is the secret word mentioned in the text? Answer with the word only."

print("tokens in the text:", len(tokenizer.encode(long_text)))

tokens in the text: 1622


In [11]:
# WRONG: a 512-token window; the start of the text is cut off, the model never sees the needle
response_small = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": long_text + "\n\n" + question}],
    think=False,
    options={"temperature": 0, "num_ctx": 512, "num_predict": 20},
)

print("num_ctx=512  ->", response_small.message.content.strip())

num_ctx=512  -> models


In [12]:
# RIGHT: a window large enough for the whole text
response_large = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": long_text + "\n\n" + question}],
    think=False,
    options={"temperature": 0, "num_ctx": 4096, "num_predict": 20},
)

print("num_ctx=4096 ->", response_large.message.content.strip())

num_ctx=4096 -> PELICAN


<div style="text-align: justify; max-width: 90ch">

With the 512-token window the model answers something other than `PELICAN`: in one example run `models`, a word from the notes it could see.
The prompt did not fit, so Ollama **truncated** it: it dropped the beginning of the text, needle included. Nothing warned you.
The only trace is in the reply: `response_small.prompt_eval_count` is at most 512, the window, while the whole prompt has over 1 600 tokens.
Overflow is silent, which is why lecture 2 counts tokens before sending documents to a model.

</div>

<div style="text-align: justify; max-width: 90ch">

Each model also has a maximum window. `ollama show qwen3:1.7b` in a terminal prints it as `context length 40960`. Ollama's default is smaller, to save memory:
4096 tokens unless the GPU has 24 GB of VRAM or more ([context length](https://docs.ollama.com/context-length)). `ollama ps` shows the window of each loaded model in its `CONTEXT` column.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Temperature

Ask a chat app the same question twice and you may get different answers. Variety helps a brainstorm and hurts a test, so you need a way to control it.

At each step the model outputs a probability for *every* token in its vocabulary, and Ollama then **samples**: it draws the next token at random, weighted by those probabilities.
**Temperature** reshapes them before the draw: at `0` the most likely token is picked every time; higher values flatten the probabilities, so less likely tokens get a chance and we get increased randomness.

Without the option, Ollama uses the model's default, `0.6` for `qwen3:1.7b` (`ollama show` lists it under Parameters), so every call in this course sets it.

</div>

In [13]:
prompt = "Suggest a name for a coffee shop run by data scientists. Answer with the name only."

rows = []

# 0: always the most likely token; 1: draw from the model's own probabilities
for temperature in [0.0, 1.0]:
    # Ask the model 3 times for each temperature to see the variability in answers
    for run in range(3):
        answer = client.chat(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            think=False,
            options={"temperature": temperature, "num_predict": 20},
        ).message.content.strip()
        rows.append({"temperature": temperature, "run": run + 1, "answer": answer})

pd.DataFrame(rows)

,temperature,run,answer
0,0.0,1,DataCafé
1,0.0,2,DataCafé
2,0.0,3,DataCafé
3,1.0,1,DataCafé
4,1.0,2,DataBrew
5,1.0,3,DataCafé


<div style="text-align: justify; max-width: 90ch">

At `0` the three runs should agree (see the note below); at `1` they can differ, and your run may show other names than the table above.

### ⏸ Pause point

Which setting would you ship in a customer-facing product, and which in a brainstorming tool? In this course we use `temperature=0` almost everywhere, for the same reason you set `random_state=42` in scikit-learn:
when a number changes, we want to know it was our change, not luck. (Temperature 0 is *nearly* deterministic; on some hardware two runs can still differ.)

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- Ollama is a **local model server**; the `ollama` package is its client: `client.chat` asks a model, `client.ps()` shows where it runs.
- Local buys privacy, no bill and a stable model, at the cost of model size and speed.
- An LLM predicts the next **token**; the reply reports tokens in, tokens out, time, and why it stopped.
- The tokenizer decides the cost: Greek needs several times more tokens than English.
- The **context window** is a hard, silent limit measured in tokens; `num_ctx` sets it per call.
- **Temperature** trades reproducibility for variety; the course uses `0`.

**Next: `lec_01b`**, choosing a model. Size is only one of the things that decide which model you run; the requirements of the job are more important.

</div>